# SOC Crosswalk Rebuild

This notebook rebuilds the SOC Felten crosswalk in a notebook-native workflow.

We keep the process in distinct stages:

1. Raw Appendix A bridge baseline
2. Accepted suggested joins
3. Combined notebook crosswalk
4. Manual review and notebook locks
5. Final canonical SOC join table


## 1. Setup

Load packages and define repo-relative paths used throughout the rebuild.


In [1]:
from __future__ import annotations

from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)

NOTEBOOK_ROOT = Path.cwd()
REPO_ROOT = NOTEBOOK_ROOT
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent

DB_PATH = REPO_ROOT / "foundations" / "etl" / "data" / "duckdb" / "patterns_in_place.duckdb"
FELTEN_WORKBOOK_PATH = REPO_ROOT / "metro-deep-dive" / "metro-area-explorer" / "industry" / "reference_data" / "AIOE_DataAppendix.xlsx"
AUDIT_ROOT = REPO_ROOT / "metro-deep-dive" / "metro-area-explorer" / "industry" / "outputs" / "national" / "d6_coverage_review"
SOC_CROSSWALK_PATH = AUDIT_ROOT / "soc_2010_to_2018_crosswalk.xlsx"
SOC_RECOMMENDATIONS_PATH = AUDIT_ROOT / "recommended_felten_soc_overrides_initial.csv"
SOC_STEP3_REVIEW_BASE_PATH = REPO_ROOT / "metro-deep-dive" / "analysis_program" / "01_ai_inversion" / "outputs" / "soc_crosswalk_step3_review_base.csv"
SOC_FINAL_CROSSWALK_OUTPUT_PATH = REPO_ROOT / "metro-deep-dive" / "analysis_program" / "01_ai_inversion" / "outputs" / "soc_felten_join_reference.csv"

display(pd.DataFrame([
    {"path": "DuckDB", "value": str(DB_PATH.relative_to(REPO_ROOT))},
    {"path": "Felten workbook", "value": str(FELTEN_WORKBOOK_PATH.relative_to(REPO_ROOT))},
    {"path": "SOC 2010-2018 bridge", "value": str(SOC_CROSSWALK_PATH.relative_to(REPO_ROOT))},
    {"path": "Accepted suggested joins", "value": str(SOC_RECOMMENDATIONS_PATH.relative_to(REPO_ROOT))},
    {"path": "Step 3 review base output", "value": str(SOC_STEP3_REVIEW_BASE_PATH.relative_to(REPO_ROOT))},
]))


,path,value
0,DuckDB,foundations/etl/data/duckdb/patterns_in_place....
1,Felten workbook,metro-deep-dive/metro-area-explorer/industry/r...
2,SOC 2010-2018 bridge,metro-deep-dive/metro-area-explorer/industry/o...
3,Accepted suggested joins,metro-deep-dive/metro-area-explorer/industry/o...
4,Step 3 review base output,metro-deep-dive/analysis_program/01_ai_inversi...


## 2. Raw Appendix A Bridge Baseline

Start from raw Felten Appendix A and the official SOC version bridge, then compare that bridge to the live detailed SOC surface.


In [2]:
# Read the raw Felten Appendix A occupation scores.
# These are the scored 2010 SOC occupations that anchor the rest of the workflow.
appendix_a = pd.read_excel(FELTEN_WORKBOOK_PATH, sheet_name="Appendix A").rename(
    columns={"SOC Code": "felten_soc_code_2010", "Occupation Title": "felten_soc_title_2010", "AIOE": "aioe_score"}
).copy()
appendix_a["felten_soc_code_2010"] = appendix_a["felten_soc_code_2010"].astype(str).str.strip()
appendix_a["felten_soc_title_2010"] = appendix_a["felten_soc_title_2010"].astype(str).str.strip()
appendix_a["aioe_score"] = pd.to_numeric(appendix_a["aioe_score"], errors="coerce")
appendix_a = appendix_a.dropna(subset=["felten_soc_code_2010"]).reset_index(drop=True)

# Read the official SOC version bridge so we can compare modern live SOC codes back to Felten's static 2010 universe.
soc_version_crosswalk = pd.read_excel(SOC_CROSSWALK_PATH, skiprows=8).rename(
    columns={
        "2010 SOC Code": "soc_2010_code",
        "2010 SOC Title": "soc_2010_title",
        "2018 SOC Code": "soc_2018_code",
        "2018 SOC Title": "soc_2018_title",
    }
).copy()
for col in ["soc_2010_code", "soc_2010_title", "soc_2018_code", "soc_2018_title"]:
    soc_version_crosswalk[col] = soc_version_crosswalk[col].astype(str).str.strip()
soc_version_crosswalk = soc_version_crosswalk.dropna(subset=["soc_2010_code", "soc_2018_code"]).reset_index(drop=True)

# Build one reusable Felten base table that carries the scored 2010 occupation and any linked 2018 SOC code.
felten_soc_base = soc_version_crosswalk.merge(
    appendix_a,
    left_on="soc_2010_code",
    right_on="felten_soc_code_2010",
    how="left",
    validate="many_to_one",
)

display(pd.DataFrame([
    {"input": "Felten Appendix A occupations", "rows": len(appendix_a)},
    {"input": "SOC 2010-to-2018 crosswalk rows", "rows": len(soc_version_crosswalk)},
    {"input": "Felten SOC base rows", "rows": len(felten_soc_base)},
    {"input": "Felten Appendix A rows with missing score", "rows": int(appendix_a["aioe_score"].isna().sum())},
]))
display(felten_soc_base.head(20))


,input,rows
0,Felten Appendix A occupations,774
1,SOC 2010-to-2018 crosswalk rows,900
2,Felten SOC base rows,900
3,Felten Appendix A rows with missing score,0


,soc_2010_code,soc_2010_title,soc_2018_code,soc_2018_title,felten_soc_code_2010,felten_soc_title_2010,aioe_score
0,11-1011,Chief Executives,11-1011,Chief Executives,11-1011,Chief Executives,1.334246
1,11-1021,General and Operations Managers,11-1021,General and Operations Managers,11-1021,General and Operations Managers,0.574877
2,11-1031,Legislators,11-1031,Legislators,NaN,NaN,NaN
3,11-2011,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers,1.294387
4,11-2021,Marketing Managers,11-2021,Marketing Managers,11-2021,Marketing Managers,1.315032
5,11-2022,Sales Managers,11-2022,Sales Managers,11-2022,Sales Managers,1.266280
6,11-2031,Public Relations and Fundraising Managers (#),11-2032,Public Relations Managers,11-2031,Public Relations Managers,1.293689
7,11-2031,Public Relations and Fundraising Managers (#),11-2033,Fundraising Managers,11-2031,Public Relations Managers,1.293689
8,11-3011,Administrative Services Managers (#),11-3012,Administrative Services Managers,11-3011,Administrative Services Managers,0.739828
9,11-3011,Administrative Services Managers (#),11-3013,Facilities Managers,11-3011,Administrative Services Managers,0.739828


In [3]:
# Pull the live detailed SOC surface by year so we can measure both yearly coverage and total coverage.
soc_reference_sql = """
WITH soc_base AS (
    SELECT
        soc_code,
        year,
        any_value(soc_title) AS soc_title,
        SUM(employment) AS sector_employment,
        COUNT(DISTINCT geo_id) AS metros
    FROM silver.bls_oews
    WHERE geo_level = 'cbsa'
      AND o_group = 'detailed'
    GROUP BY 1, 2
)
SELECT
    soc_code,
    year,
    soc_title,
    sector_employment,
    SUM(sector_employment) OVER (PARTITION BY year) AS total_employment,
    sector_employment / SUM(sector_employment) OVER (PARTITION BY year) AS sector_weight,
    metros,
    MAX(metros) OVER (PARTITION BY year) AS max_metros
FROM soc_base
ORDER BY year, sector_employment DESC, soc_code
"""

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    soc_reference_yearly = con.execute(soc_reference_sql).fetchdf()

soc_reference_codes = (
    soc_reference_yearly.sort_values(["year", "sector_employment"], ascending=[False, False])
    .drop_duplicates(subset=["soc_code"])[["soc_code", "soc_title"]]
    .rename(columns={"soc_title": "soc_title_ours"})
    .reset_index(drop=True)
)

display(soc_reference_yearly.head(20))
display(pd.DataFrame([{
    "rows": len(soc_reference_yearly),
    "soc_codes": soc_reference_yearly["soc_code"].nunique(),
    "year_min": int(soc_reference_yearly["year"].min()),
    "year_max": int(soc_reference_yearly["year"].max()),
}]))


,soc_code,year,soc_title,sector_employment,total_employment,sector_weight,metros,max_metros
0,31-1120,2025,Home Health and Personal Care Aides,3887140.0,130416820.0,0.029806,392,393
1,41-2031,2025,Retail Salespersons,3439720.0,130416820.0,0.026375,393,393
2,35-3023,2025,Fast Food and Counter Workers,3408200.0,130416820.0,0.026133,393,393
3,29-1141,2025,Registered Nurses,2985110.0,130416820.0,0.022889,388,393
4,11-1021,2025,General and Operations Managers,2949930.0,130416820.0,0.022619,392,393
5,41-2011,2025,Cashiers,2615670.0,130416820.0,0.020056,393,393
6,53-7062,2025,"Laborers and Freight, Stock, and Material Move...",2565600.0,130416820.0,0.019672,392,393
7,53-7065,2025,Stockers and Order Fillers,2458870.0,130416820.0,0.018854,393,393
8,43-4051,2025,Customer Service Representatives,2195480.0,130416820.0,0.016834,393,393
9,43-9061,2025,"Office Clerks, General",2086330.0,130416820.0,0.015997,392,393


,rows,soc_codes,year_min,year_max
0,824,824,2025,2025


In [4]:
# Join live SOC codes to the raw Felten bridge.
# We keep the full candidate table so we can inspect splits where one 2018 SOC code points back to multiple 2010 Felten occupations.
soc_raw_candidates = soc_reference_codes.merge(
    felten_soc_base[[
        "soc_2018_code",
        "soc_2018_title",
        "felten_soc_code_2010",
        "felten_soc_title_2010",
        "aioe_score",
    ]].rename(
        columns={
            "soc_2018_code": "felten_soc_code",
            "soc_2018_title": "felten_soc_title_selected_version",
            "felten_soc_title_2010": "felten_soc_title",
        }
    ),
    left_on="soc_code",
    right_on="felten_soc_code",
    how="left",
)

soc_raw_candidates["raw_candidate_count"] = soc_raw_candidates.groupby("soc_code")["felten_soc_code_2010"].transform(lambda values: values.notna().sum())

soc_raw_status = (
    soc_reference_yearly[["year", "soc_code", "soc_title", "sector_employment", "total_employment", "sector_weight"]]
    .drop_duplicates()
    .merge(
        soc_raw_candidates.groupby("soc_code", as_index=False).agg(
            raw_candidate_count=("felten_soc_code_2010", lambda values: int(values.notna().sum())),
            raw_scored_match_flag=("aioe_score", lambda values: bool(values.notna().any())),
        ),
        on="soc_code",
        how="left",
    )
)
soc_raw_status["raw_candidate_count"] = soc_raw_status["raw_candidate_count"].fillna(0).astype(int)
soc_raw_status["raw_bridge_match_flag"] = soc_raw_status["raw_candidate_count"] > 0
soc_raw_status["raw_scored_match_flag"] = soc_raw_status["raw_scored_match_flag"].fillna(False)

raw_duplicate_summary = (
    soc_raw_candidates.loc[soc_raw_candidates["raw_candidate_count"] > 1, ["soc_code", "soc_title_ours", "raw_candidate_count"]]
    .drop_duplicates()
    .sort_values(["raw_candidate_count", "soc_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

def summarize_soc_coverage(df: pd.DataFrame, flag_col: str) -> pd.DataFrame:
    yearly = (
        df.groupby("year", as_index=False)
        .agg(
            total_codes=("soc_code", "size"),
            matched_codes=(flag_col, "sum"),
            total_weight=("sector_weight", "sum"),
            matched_weight=("sector_weight", lambda values: float(values[df.loc[values.index, flag_col]].sum())),
        )
    )
    yearly["coverage_pct"] = yearly["matched_codes"] / yearly["total_codes"]
    yearly["weighted_coverage_pct"] = yearly["matched_weight"] / yearly["total_weight"]

    overall = pd.DataFrame([
        {
            "year": "all_years",
            "total_codes": len(df),
            "matched_codes": int(df[flag_col].sum()),
            "total_weight": float(df["sector_weight"].sum()),
            "matched_weight": float(df.loc[df[flag_col], "sector_weight"].sum()),
        }
    ])
    overall["coverage_pct"] = overall["matched_codes"] / overall["total_codes"]
    overall["weighted_coverage_pct"] = overall["matched_weight"] / overall["total_weight"]

    return pd.concat([yearly, overall], ignore_index=True)[["year", "coverage_pct", "weighted_coverage_pct"]]

soc_raw_bridge_coverage = summarize_soc_coverage(soc_raw_status, "raw_bridge_match_flag")
soc_raw_scored_coverage = summarize_soc_coverage(soc_raw_status, "raw_scored_match_flag")
latest_year = int(soc_reference_yearly["year"].max())
soc_raw_missing_latest = (
    soc_raw_status.loc[
        (soc_raw_status["year"] == latest_year) & (~soc_raw_status["raw_scored_match_flag"]),
        ["year", "soc_code", "soc_title", "sector_weight"],
    ]
    .sort_values(["sector_weight", "soc_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

display(pd.DataFrame([
    {"metric": "Live SOC codes in reference surface", "value": soc_reference_codes["soc_code"].nunique()},
    {"metric": "Live SOC codes with multiple raw Felten candidates", "value": raw_duplicate_summary["soc_code"].nunique()},
]))
display(raw_duplicate_summary.head(30))
display(soc_raw_bridge_coverage.style.format({"coverage_pct": "{:.1%}", "weighted_coverage_pct": "{:.1%}"}))
display(soc_raw_scored_coverage.style.format({"coverage_pct": "{:.1%}", "weighted_coverage_pct": "{:.1%}"}))
display(soc_raw_missing_latest)


,metric,value
0,Live SOC codes in reference surface,824
1,Live SOC codes with multiple raw Felten candid...,22


,soc_code,soc_title_ours,raw_candidate_count
0,13-1082,Project Management Specialists,3
1,15-1253,Software Quality Assurance Analysts and Testers,3
2,13-2054,Financial Risk Specialists,2
3,15-1243,Database Architects,2
4,15-1252,Software Developers,2
5,15-1255,Web and Digital Interface Designers,2
6,15-1299,"Computer Occupations, All Other",2
7,19-4044,Hydrologic Technicians,2
8,25-4022,Librarians and Media Collections Specialists,2
9,27-3023,"News Analysts, Reporters, and Journalists",2


,year,coverage_pct,weighted_coverage_pct
0,2025,93.0%,92.4%
1,all_years,93.0%,92.4%


,year,coverage_pct,weighted_coverage_pct
0,2025,93.0%,92.4%
1,all_years,93.0%,92.4%


,year,soc_code,soc_title,sector_weight
0,2025,31-1120,Home Health and Personal Care Aides,0.029806
1,2025,25-9045,"Teaching Assistants, Except Postsecondary",0.009204
2,2025,51-2090,Miscellaneous Assemblers and Fabricators,0.008596
3,2025,53-1047,First-Line Supervisors of Transportation and M...,0.004179
4,2025,13-1020,Buyers and Purchasing Agents,0.003320
5,2025,21-1018,"Substance Abuse, Behavioral Disorder, and Ment...",0.003305
6,2025,29-2010,Clinical Laboratory Technologists and Technicians,0.002264
7,2025,15-2051,Data Scientists,0.001709
8,2025,25-2052,"Special Education Teachers, Kindergarten and E...",0.001707
9,2025,51-2028,"Electrical, Electronic, and Electromechanical ...",0.001565


## 3. Accepted Suggested Joins

Load the accepted suggested joins as a distinct reviewed layer that sits on top of the raw bridge.


In [5]:
# Read the first-pass recommendation file and keep only the rows we explicitly accept.
# These rows are still notebook-owned review decisions, but they are distinct from the raw bridge itself.
soc_recommendations = pd.read_csv(SOC_RECOMMENDATIONS_PATH, dtype=str)
soc_recommendations_accepted = soc_recommendations.loc[
    soc_recommendations["recommend_match"].astype(str).str.lower() == "true"
].copy()

display(soc_recommendations_accepted.head(20))
display(pd.DataFrame([
    {"metric": "Accepted suggested join rows", "value": len(soc_recommendations_accepted)},
    {"metric": "Accepted suggested join codes", "value": soc_recommendations_accepted["our_code"].nunique()},
]))


,our_code,our_name,our_weight,our_share_of_total,recommended_felten_code,recommended_felten_name,recommended_score,recommended_felten_score,recommend_match,confidence,rationale
0,31-1120,Home Health and Personal Care Aides,4305810.0,0.027911941652262795,31-1011,Home Health Aides,0.6538461538461539,-0.644942,True,medium,Same SOC major group with moderate title align...
1,15-1252,Software Developers,1687900.0,0.010941626851824482,15-1132,"Software Developers, Applications (#)",0.7450980392156863,NaN,True,medium,Same SOC major group with moderate title align...
2,31-1131,Nursing Assistants,1448920.0,0.009392465180487902,31-1014,Nursing Assistants,1.0,NaN,True,high,Same SOC major group with strong title alignment.
3,25-9045,"Teaching Assistants, Except Postsecondary",1420390.0,0.009207522580758918,25-1112,"Law Teachers, Postsecondary",0.6060606060606061,1.35645,True,medium,Same SOC major group with moderate title align...
4,51-2090,Miscellaneous Assemblers and Fabricators,1405050.0,0.009108082711153497,51-2091,Fiberglass Laminators and Fabricators,0.6493506493506493,-1.216496,True,medium,Same SOC major group with moderate title align...
5,15-1232,Computer User Support Specialists,717190.0,0.004649105611623912,15-1151,Computer User Support Specialists,1.0,NaN,True,high,Same SOC major group with strong title alignment.
6,53-1047,First-Line Supervisors of Transportation and M...,623640.0,0.004042677984401812,53-1031,First-Line Supervisors/Managers of Transportat...,0.7047619047619048,0.1466244999999999,True,medium,Same SOC major group with moderate title align...
7,15-1211,Computer Systems Analysts,519540.0,0.0033678611378617756,15-1121,Computer Systems Analysts,1.0,NaN,True,high,Same SOC major group with strong title alignment.
8,21-1018,"Substance Abuse, Behavioral Disorder, and Ment...",487020.0,0.0031570538002106516,21-1011,Substance Abuse and Behavioral Disorder Counse...,0.8070175438596491,1.030841,True,high,Same SOC major group with strong title alignment.
9,15-1299,"Computer Occupations, All Other",435390.0,0.0028223679809324374,15-1199,"Computer Occupations, All Other (#)",1.0,NaN,True,high,Same SOC major group with strong title alignment.


,metric,value
0,Accepted suggested join rows,116
1,Accepted suggested join codes,116


## 4. Combined Notebook Crosswalk

Combine the raw scored bridge and the accepted suggested joins into one notebook-owned intermediate crosswalk.


In [6]:
# Build one notebook-owned SOC crosswalk for review.
# This step is intentionally simple:
# 1. keep the raw bridge rows that already resolve to a Felten score
# 2. layer the accepted suggested joins on top
# 3. anchor the result back to the full live SOC code universe so step 5 can review the remaining gaps

soc_raw_matched_rows = (
    soc_raw_candidates.loc[
        soc_raw_candidates["aioe_score"].notna(),
        [
            "soc_code",
            "soc_title_ours",
            "felten_soc_code",
            "felten_soc_title_selected_version",
            "aioe_score",
            "raw_candidate_count",
        ],
    ]
    .sort_values(["soc_code", "aioe_score"], ascending=[True, False], kind="mergesort")
    .drop_duplicates(subset=["soc_code"], keep="first")
    .rename(
        columns={
            "felten_soc_code": "selected_felten_code",
            "felten_soc_title_selected_version": "selected_felten_name",
        }
    )
    .reset_index(drop=True)
)
soc_raw_matched_rows["match_basis"] = soc_raw_matched_rows["raw_candidate_count"].map(
    lambda value: "raw_crosswalk_deduplicated" if int(value) > 1 else "raw_crosswalk_single_match"
)
soc_raw_matched_rows["manual_notes"] = pd.NA
soc_raw_matched_rows["review_source"] = "notebook_raw_soc_bridge"
soc_raw_matched_rows = soc_raw_matched_rows.drop(columns=["raw_candidate_count"])

soc_accepted_join_rows = (
    soc_recommendations_accepted[[
        "our_code",
        "our_name",
        "recommended_felten_code",
        "recommended_felten_name",
        "recommended_felten_score",
        "rationale",
    ]]
    .rename(
        columns={
            "our_code": "soc_code",
            "our_name": "soc_title_ours",
            "recommended_felten_code": "selected_felten_code",
            "recommended_felten_name": "selected_felten_name",
            "recommended_felten_score": "aioe_score",
            "rationale": "manual_notes",
        }
    )
    .copy()
)
soc_accepted_join_rows["aioe_score"] = pd.to_numeric(soc_accepted_join_rows["aioe_score"], errors="coerce")
soc_accepted_join_rows["match_basis"] = "accepted_suggested_join"
soc_accepted_join_rows["review_source"] = "recommended_felten_soc_overrides_initial"

soc_step3_matched_base = pd.concat(
    [soc_raw_matched_rows, soc_accepted_join_rows],
    ignore_index=True,
).drop_duplicates(subset=["soc_code"], keep="last").sort_values("soc_code", kind="mergesort").reset_index(drop=True)

soc_latest_weights = (
    soc_reference_yearly.loc[soc_reference_yearly["year"] == latest_year, ["soc_code", "sector_employment", "sector_weight"]]
    .drop_duplicates(subset=["soc_code"])
    .rename(columns={"sector_employment": "latest_year_employment", "sector_weight": "latest_year_weight"})
)

soc_step3_review_base = (
    soc_reference_codes.merge(
        soc_step3_matched_base.drop(columns=["soc_title_ours"]),
        on="soc_code",
        how="left",
        validate="one_to_one",
    )
    .merge(
        soc_latest_weights,
        on="soc_code",
        how="left",
        validate="one_to_one",
    )
    .sort_values(["latest_year_weight", "soc_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)
soc_step3_review_base["step3_match_flag"] = soc_step3_review_base["aioe_score"].notna()

soc_step3_coverage = summarize_soc_coverage(
    soc_reference_yearly[["year", "soc_code", "sector_weight"]]
    .drop_duplicates()
    .merge(
        soc_step3_review_base[["soc_code", "step3_match_flag"]],
        on="soc_code",
        how="left",
        validate="many_to_one",
    )
    .assign(step3_match_flag=lambda df: df["step3_match_flag"].fillna(False)),
    "step3_match_flag",
)

soc_step3_missing_latest = (
    soc_step3_review_base.loc[
        ~soc_step3_review_base["step3_match_flag"],
        ["soc_code", "soc_title_ours", "latest_year_employment", "latest_year_weight"],
    ]
    .sort_values(["latest_year_weight", "soc_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

SOC_STEP3_REVIEW_BASE_PATH.parent.mkdir(parents=True, exist_ok=True)
soc_step3_review_base.to_csv(SOC_STEP3_REVIEW_BASE_PATH, index=False)

display(pd.DataFrame([
    {"metric": "Raw matched rows kept", "value": len(soc_raw_matched_rows)},
    {"metric": "Accepted suggested joins kept", "value": len(soc_accepted_join_rows)},
    {"metric": "Step 3 review-base rows", "value": len(soc_step3_review_base)},
    {"metric": "Latest-year unresolved after step 3", "value": len(soc_step3_missing_latest)},
    {"metric": "Step 3 review base output", "value": str(SOC_STEP3_REVIEW_BASE_PATH.relative_to(REPO_ROOT))},
]))
display(soc_step3_coverage.style.format({"coverage_pct": "{:.1%}", "weighted_coverage_pct": "{:.1%}"}))
display(soc_step3_review_base.head(30))
display(soc_step3_missing_latest.head(50))


,metric,value
0,Raw matched rows kept,766
1,Accepted suggested joins kept,116
2,Step 3 review-base rows,824
3,Latest-year unresolved after step 3,114
4,Step 3 review base output,metro-deep-dive/analysis_program/01_ai_inversi...


,year,coverage_pct,weighted_coverage_pct
0,2025,86.2%,91.9%
1,all_years,86.2%,91.9%


,soc_code,soc_title_ours,selected_felten_code,selected_felten_name,aioe_score,match_basis,manual_notes,review_source,latest_year_employment,latest_year_weight,step3_match_flag
0,31-1120,Home Health and Personal Care Aides,31-1011,Home Health Aides,-0.644942,accepted_suggested_join,Same SOC major group with moderate title align...,recommended_felten_soc_overrides_initial,3887140.0,0.029806,True
1,41-2031,Retail Salespersons,41-2031,Retail Salespersons,0.087307,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge,3439720.0,0.026375,True
2,35-3023,Fast Food and Counter Workers,35-3023,Fast Food and Counter Workers (##),-0.485096,raw_crosswalk_deduplicated,NaN,notebook_raw_soc_bridge,3408200.0,0.026133,True
3,29-1141,Registered Nurses,29-1141,Registered Nurses,0.229410,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge,2985110.0,0.022889,True
4,11-1021,General and Operations Managers,11-1021,General and Operations Managers,0.574877,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge,2949930.0,0.022619,True
5,41-2011,Cashiers,41-2011,Cashiers,-0.248089,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge,2615670.0,0.020056,True
6,53-7062,"Laborers and Freight, Stock, and Material Move...",53-7062,"Laborers and Freight, Stock, and Material Move...",-1.709183,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge,2565600.0,0.019672,True
7,53-7065,Stockers and Order Fillers,53-7065,Stockers and Order Fillers,-0.789295,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge,2458870.0,0.018854,True
8,43-4051,Customer Service Representatives,43-4051,Customer Service Representatives,0.956252,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge,2195480.0,0.016834,True
9,43-9061,"Office Clerks, General",43-9061,"Office Clerks, General",0.988899,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge,2086330.0,0.015997,True


,soc_code,soc_title_ours,latest_year_employment,latest_year_weight
0,15-1252,Software Developers,1441640.0,0.011054
1,31-1131,Nursing Assistants,1217690.0,0.009337
2,15-1232,Computer User Support Specialists,599640.0,0.004598
3,15-1211,Computer Systems Analysts,441640.0,0.003386
4,13-1020,Buyers and Purchasing Agents,433030.0,0.003320
5,15-1299,"Computer Occupations, All Other",350320.0,0.002686
6,53-3051,"Bus Drivers, School",322040.0,0.002469
7,29-1229,"Physicians, All Other",300910.0,0.002307
8,15-1244,Network and Computer Systems Administrators,267080.0,0.002048
9,15-2051,Data Scientists,222920.0,0.001709


## 5. Manual Review

Review the remaining unresolved SOC codes and add notebook manual locks after inspection.


In [7]:
# QA helper for manual SOC review.
# Change REVIEW_SOC_CODE to any live SOC code you want to inspect.
# This keeps the manual review surface intentionally simple:
# 1. the target live code from our step 3 review base
# 2. the live latest-year rows in the same SOC major group
# 3. the nearby Felten Appendix A rows in the same SOC major group

REVIEW_SOC_CODE = "31-1120"
FAMILY_DIGITS = 2

review_prefix = str(REVIEW_SOC_CODE)[:FAMILY_DIGITS]

review_live_code = (
    soc_step3_review_base.loc[soc_step3_review_base["soc_code"] == REVIEW_SOC_CODE]
    .reset_index(drop=True)
)

review_live_family_latest = (
    soc_step3_review_base.loc[
        soc_step3_review_base["soc_code"].astype(str).str[:FAMILY_DIGITS] == review_prefix,
        [
            "soc_code",
            "soc_title_ours",
            "latest_year_employment",
            "latest_year_weight",
            "selected_felten_code",
            "selected_felten_name",
            "aioe_score",
            "step3_match_flag",
        ],
    ]
    .sort_values(["latest_year_weight", "soc_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

review_family_appendix = (
    felten_soc_base.loc[
        felten_soc_base["felten_soc_code_2010"].astype(str).str[:FAMILY_DIGITS] == review_prefix,
        ["felten_soc_code_2010", "felten_soc_title_2010", "aioe_score", "soc_2018_code", "soc_2018_title"],
    ]
    .drop_duplicates()
    .sort_values(["felten_soc_code_2010", "soc_2018_code"], kind="mergesort")
    .reset_index(drop=True)
)

display(pd.DataFrame([
    {"review_soc_code": REVIEW_SOC_CODE, "family_prefix": review_prefix, "family_digits": FAMILY_DIGITS},
]))
display(review_live_code)
display(review_live_family_latest)
display(review_family_appendix)


,review_soc_code,family_prefix,family_digits
0,31-1120,31,2


,soc_code,soc_title_ours,selected_felten_code,selected_felten_name,aioe_score,match_basis,manual_notes,review_source,latest_year_employment,latest_year_weight,step3_match_flag
0,31-1120,Home Health and Personal Care Aides,31-1011,Home Health Aides,-0.644942,accepted_suggested_join,Same SOC major group with moderate title align...,recommended_felten_soc_overrides_initial,3887140.0,0.029806,True


,soc_code,soc_title_ours,latest_year_employment,latest_year_weight,selected_felten_code,selected_felten_name,aioe_score,step3_match_flag
0,31-1120,Home Health and Personal Care Aides,3887140.0,0.029806,31-1011,Home Health Aides,-0.644942,True
1,31-1131,Nursing Assistants,1217690.0,0.009337,31-1014,Nursing Assistants,NaN,False
2,31-9092,Medical Assistants,728920.0,0.005589,31-9092,Medical Assistants,0.149792,True
3,31-9091,Dental Assistants,349720.0,0.002682,31-9091,Dental Assistants,-0.526876,True
4,31-9097,Phlebotomists,123240.0,0.000945,31-9097,Phlebotomists,-0.277848,True
5,31-9096,Veterinary Assistants and Laboratory Animal Ca...,104650.0,0.000802,31-9096,Veterinary Assistants and Laboratory Animal Ca...,-0.773582,True
6,31-9099,"Healthcare Support Workers, All Other",94530.0,0.000725,31-9099,"Healthcare Support Workers, All Other",0.166008,True
7,31-2021,Physical Therapist Assistants,94000.0,0.000721,31-2021,Physical Therapist Assistants,-0.530853,True
8,31-9011,Massage Therapists,88270.0,0.000677,31-9011,Massage Therapists,-1.410233,True
9,31-9093,Medical Equipment Preparers,65990.0,0.000506,31-9093,Medical Equipment Preparers,-0.730865,True


,felten_soc_code_2010,felten_soc_title_2010,aioe_score,soc_2018_code,soc_2018_title
0,31-1011,Home Health Aides,-0.644942,31-1121,Home Health Aides
1,31-1013,Psychiatric Aides,-0.496565,31-1133,Psychiatric Aides
2,31-1014,Nursing Assistants,-0.904021,31-1131,Nursing Assistants
3,31-1015,Orderlies,-1.679232,31-1132,Orderlies
4,31-2011,Occupational Therapist Assistants,-0.315376,31-2011,Occupational Therapy Assistants
5,31-2012,Occupational Therapist Aides,-0.773432,31-2012,Occupational Therapy Aides
6,31-2021,Physical Therapist Assistants,-0.530853,31-2021,Physical Therapist Assistants
7,31-2022,Physical Therapist Aides,-0.882327,31-2022,Physical Therapist Aides
8,31-9011,Massage Therapists,-1.410233,31-9011,Massage Therapists
9,31-9091,Dental Assistants,-0.526876,31-9091,Dental Assistants


In [8]:
# These are the notebook-level SOC manual lock decisions we make after reviewing the remaining step 3 gaps.
# These reflect the validated review decisions already developed in the main AI inversion notebook.
soc_notebook_locked_overrides = pd.DataFrame([
    {"soc_code": "31-1120", "selected_felten_code": "31-1121", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "25-9045", "selected_felten_code": "25-9049", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "51-2090", "selected_felten_code": "51-2092", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "53-1047", "selected_felten_code": "53-1043", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "13-1020", "selected_felten_code": "13-1022", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "21-1018", "selected_felten_code": "21-1023", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "29-2010", "selected_felten_code": "29-2012", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "25-2052", "selected_felten_code": "25-2056", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "51-2028", "selected_felten_code": "51-2023", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "39-7010", "selected_felten_code": "39-7011", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "13-2020", "selected_felten_code": "13-2023", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "47-4090", "selected_felten_code": "47-4099", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "15-2051", "selected_felten_code": "15-1121", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "43-9199", "selected_felten_code": "43-9061", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "43-4199", "selected_felten_code": "43-4171", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "25-1199", "selected_felten_code": "25-1194", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "25-9099", "selected_felten_code": "25-9049", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "21-1099", "selected_felten_code": "25-1094", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "41-9099", "selected_felten_code": "41-9091", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "33-1091", "selected_felten_code": "33-1011", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "35-9099", "selected_felten_code": "35-3041", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "21-1029", "selected_felten_code": "21-1022", "manual_notes": "Notebook lock for missing-score review"},
    {"soc_code": "39-9099", "selected_felten_code": "31-1122", "manual_notes": "Notebook lock for missing-score review"},
])
soc_notebook_locked_overrides["review_source"] = "notebook_locked_missing_score"
soc_notebook_locked_overrides["match_basis"] = "manual_override"

display(pd.DataFrame([
    {"input": "Notebook SOC manual locks", "rows": len(soc_notebook_locked_overrides)},
]))
display(soc_notebook_locked_overrides)


,input,rows
0,Notebook SOC manual locks,23


,soc_code,selected_felten_code,manual_notes,review_source,match_basis
0,31-1120,31-1121,Notebook lock for missing-score review,notebook_locked_missing_score,manual_override
1,25-9045,25-9049,Notebook lock for missing-score review,notebook_locked_missing_score,manual_override
2,51-2090,51-2092,Notebook lock for missing-score review,notebook_locked_missing_score,manual_override
3,53-1047,53-1043,Notebook lock for missing-score review,notebook_locked_missing_score,manual_override
4,13-1020,13-1022,Notebook lock for missing-score review,notebook_locked_missing_score,manual_override
5,21-1018,21-1023,Notebook lock for missing-score review,notebook_locked_missing_score,manual_override
6,29-2010,29-2012,Notebook lock for missing-score review,notebook_locked_missing_score,manual_override
7,25-2052,25-2056,Notebook lock for missing-score review,notebook_locked_missing_score,manual_override
8,51-2028,51-2023,Notebook lock for missing-score review,notebook_locked_missing_score,manual_override
9,39-7010,39-7011,Notebook lock for missing-score review,notebook_locked_missing_score,manual_override


## 6. Final Canonical SOC Join Table

Combine raw matched rows, accepted suggested joins, and notebook manual locks into the final one-row-per-code table used downstream.


In [9]:
# Build the final canonical SOC join table from the step 3 review base plus notebook manual locks.
# This final table is the only SOC join surface we should use downstream once review is complete.

soc_final_crosswalk_base = soc_step3_review_base.merge(
    soc_notebook_locked_overrides[["soc_code", "selected_felten_code", "manual_notes", "review_source", "match_basis"]].rename(
        columns={
            "selected_felten_code": "notebook_selected_felten_code",
            "manual_notes": "notebook_manual_notes",
            "review_source": "notebook_review_source",
            "match_basis": "notebook_match_basis",
        }
    ),
    on="soc_code",
    how="left",
    validate="one_to_one",
)

soc_final_crosswalk_base["selected_felten_code_final"] = soc_final_crosswalk_base["notebook_selected_felten_code"].combine_first(
    soc_final_crosswalk_base["selected_felten_code"]
)
soc_final_crosswalk_base["manual_notes_final"] = soc_final_crosswalk_base["notebook_manual_notes"].combine_first(
    soc_final_crosswalk_base["manual_notes"]
)
soc_final_crosswalk_base["review_source_final"] = soc_final_crosswalk_base["notebook_review_source"].combine_first(
    soc_final_crosswalk_base["review_source"]
)
soc_final_crosswalk_base["match_basis_final"] = soc_final_crosswalk_base["notebook_match_basis"].combine_first(
    soc_final_crosswalk_base["match_basis"]
)

# Resolve the final selected Felten code back to both the scored 2010 occupation and the selected-version title.
felten_lookup_2010 = felten_soc_base[["felten_soc_code_2010", "felten_soc_title_2010", "aioe_score"]].drop_duplicates(
    subset=["felten_soc_code_2010"],
    keep="first",
).rename(
    columns={
        "felten_soc_code_2010": "selected_felten_code_final",
        "felten_soc_title_2010": "felten_soc_title_2010_final",
        "aioe_score": "aioe_score_2010_lookup",
    }
)
felten_lookup_2010["selected_version_title_2010_lookup"] = felten_lookup_2010["felten_soc_title_2010_final"]
felten_lookup_2010["felten_soc_code_2010_final"] = felten_lookup_2010["selected_felten_code_final"]

felten_lookup_2018 = (
    felten_soc_base[["soc_2018_code", "soc_2018_title", "felten_soc_code_2010", "felten_soc_title_2010", "aioe_score", "soc_2010_code"]]
    .dropna(subset=["soc_2018_code"])
    .sort_values(["soc_2018_code", "aioe_score", "soc_2010_code"], ascending=[True, False, True], kind="mergesort")
    .drop_duplicates(subset=["soc_2018_code"], keep="first")
    .rename(
        columns={
            "soc_2018_code": "selected_felten_code_final",
            "soc_2018_title": "selected_version_title_2018_lookup",
            "felten_soc_code_2010": "felten_soc_code_2010_from_2018_lookup",
            "felten_soc_title_2010": "felten_soc_title_2010_from_2018_lookup",
            "aioe_score": "aioe_score_2018_lookup",
        }
    )
)

soc_final_crosswalk_base = soc_final_crosswalk_base.merge(felten_lookup_2010, on="selected_felten_code_final", how="left")
soc_final_crosswalk_base = soc_final_crosswalk_base.merge(felten_lookup_2018, on="selected_felten_code_final", how="left")

soc_final_crosswalk_base["felten_soc_code_2010_final"] = soc_final_crosswalk_base["felten_soc_code_2010_final"].combine_first(
    soc_final_crosswalk_base["felten_soc_code_2010_from_2018_lookup"]
)
soc_final_crosswalk_base["felten_soc_title_final"] = soc_final_crosswalk_base["felten_soc_title_2010_final"].combine_first(
    soc_final_crosswalk_base["felten_soc_title_2010_from_2018_lookup"]
)
soc_final_crosswalk_base["felten_soc_title_selected_version_final"] = soc_final_crosswalk_base["selected_version_title_2010_lookup"].combine_first(
    soc_final_crosswalk_base["selected_version_title_2018_lookup"]
).combine_first(soc_final_crosswalk_base["selected_felten_name"])
soc_final_crosswalk_base["aioe_score_final"] = soc_final_crosswalk_base["aioe_score_2010_lookup"].combine_first(
    soc_final_crosswalk_base["aioe_score_2018_lookup"]
).combine_first(soc_final_crosswalk_base["aioe_score"])

soc_felten_join_reference = soc_final_crosswalk_base[[
    "soc_code",
    "soc_title_ours",
    "selected_felten_code_final",
    "felten_soc_title_selected_version_final",
    "felten_soc_code_2010_final",
    "felten_soc_title_final",
    "aioe_score_final",
    "match_basis_final",
    "manual_notes_final",
    "review_source_final",
]].rename(
    columns={
        "soc_code": "our_soc_code",
        "soc_title_ours": "our_name",
        "selected_felten_code_final": "felten_soc_code",
        "felten_soc_title_selected_version_final": "felten_soc_name",
        "felten_soc_title_final": "felten_soc_title",
        "aioe_score_final": "felten_score",
        "match_basis_final": "match_basis",
        "manual_notes_final": "manual_notes",
        "review_source_final": "review_source",
    }
).drop_duplicates(subset=["our_soc_code"], keep="last").sort_values("our_soc_code", kind="mergesort").reset_index(drop=True)

SOC_FINAL_CROSSWALK_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
soc_felten_join_reference.to_csv(SOC_FINAL_CROSSWALK_OUTPUT_PATH, index=False)

soc_final_status = (
    soc_reference_yearly[["year", "soc_code", "soc_title", "sector_employment", "total_employment", "sector_weight"]]
    .drop_duplicates()
    .merge(
        soc_felten_join_reference[["our_soc_code", "felten_score"]].rename(columns={"our_soc_code": "soc_code"}),
        on="soc_code",
        how="left",
        validate="many_to_one",
    )
)
soc_final_status["final_scored_match_flag"] = soc_final_status["felten_score"].notna()

soc_final_coverage = summarize_soc_coverage(soc_final_status, "final_scored_match_flag")
soc_final_missing_latest = (
    soc_final_status.loc[
        (soc_final_status["year"] == latest_year) & (~soc_final_status["final_scored_match_flag"]),
        ["year", "soc_code", "soc_title", "sector_weight"],
    ]
    .sort_values(["sector_weight", "soc_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

display(pd.DataFrame([
    {"metric": "Canonical SOC join rows", "value": len(soc_felten_join_reference)},
    {"metric": "Canonical SOC rows with score", "value": int(soc_felten_join_reference["felten_score"].notna().sum())},
    {"metric": "Notebook SOC lock rows applied", "value": int(soc_final_crosswalk_base["notebook_selected_felten_code"].notna().sum())},
    {"metric": "Latest-year unresolved after final locks", "value": len(soc_final_missing_latest)},
    {"metric": "Final join output", "value": str(SOC_FINAL_CROSSWALK_OUTPUT_PATH.relative_to(REPO_ROOT))},
]))
display(soc_final_coverage.style.format({"coverage_pct": "{:.1%}", "weighted_coverage_pct": "{:.1%}"}))
display(soc_final_missing_latest)
display(soc_felten_join_reference.head(30))


,metric,value
0,Canonical SOC join rows,824
1,Canonical SOC rows with score,788
2,Notebook SOC lock rows applied,23
3,Latest-year unresolved after final locks,36
4,Final join output,metro-deep-dive/analysis_program/01_ai_inversi...


,year,coverage_pct,weighted_coverage_pct
0,2025,95.6%,99.7%
1,all_years,95.6%,99.7%


,year,soc_code,soc_title,sector_weight
0,2025,21-1099,"Community and Social Service Specialists, All ...",0.000699
1,2025,53-3099,"Motor Vehicle Operators, All Other",0.000279
2,2025,51-3099,"Food Processing Workers, All Other",0.000257
3,2025,23-2099,"Legal Support Workers, All Other",0.000247
4,2025,43-3099,"Financial Clerks, All Other",0.000187
5,2025,21-1019,"Counselors, All Other",0.000162
6,2025,53-6032,Aircraft Service Attendants,0.000150
7,2025,47-3019,"Helpers, Construction Trades, All Other",0.000145
8,2025,33-1099,First-Line Supervisors of Protective Service W...,0.000109
9,2025,27-2099,"Entertainers and Performers, Sports and Relate...",0.000097


,our_soc_code,our_name,felten_soc_code,felten_soc_name,felten_soc_code_2010_final,felten_soc_title,felten_score,match_basis,manual_notes,review_source
0,11-1011,Chief Executives,11-1011,Chief Executives,11-1011,Chief Executives,1.334246,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge
1,11-1021,General and Operations Managers,11-1021,General and Operations Managers,11-1021,General and Operations Managers,0.574877,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge
2,11-2011,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers,1.294387,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge
3,11-2021,Marketing Managers,11-2021,Marketing Managers,11-2021,Marketing Managers,1.315032,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge
4,11-2022,Sales Managers,11-2022,Sales Managers,11-2022,Sales Managers,1.266280,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge
5,11-2032,Public Relations Managers,11-2031,Public Relations Managers,11-2031,Public Relations Managers,1.293689,accepted_suggested_join,Same SOC major group with moderate title align...,recommended_felten_soc_overrides_initial
6,11-2033,Fundraising Managers,11-2031,Public Relations Managers,11-2031,Public Relations Managers,1.293689,accepted_suggested_join,Same SOC major group with moderate title align...,recommended_felten_soc_overrides_initial
7,11-3012,Administrative Services Managers,11-3011,Administrative Services Managers,11-3011,Administrative Services Managers,0.739828,accepted_suggested_join,Same SOC major group with strong title alignment.,recommended_felten_soc_overrides_initial
8,11-3013,Facilities Managers,11-3011,Administrative Services Managers,11-3011,Administrative Services Managers,0.739828,accepted_suggested_join,Same SOC major group with moderate title align...,recommended_felten_soc_overrides_initial
9,11-3021,Computer and Information Systems Managers,11-3021,Computer and Information Systems Managers,11-3021,Computer and Information Systems Managers,1.059853,raw_crosswalk_single_match,NaN,notebook_raw_soc_bridge
